# FortiOS 8 KVM Lab - Colab Terminal Access
## Full Lab Environment + Terminal Control from Google Colab

This notebook loads ALL lab files into Colab and provides:
1. HTTP service (server)
2. CLI tool (command-line interface)
3. Web UI (browser interface)
4. Terminal access (bash commands)

---

## STEP 1: Setup Colab Environment

In [ ]:
import os
import sys
import subprocess
import time
from pathlib import Path

print("🚀 Setting up Colab environment...\n")

# Create working directory
LAB_DIR = Path('/content/fortios8_kvm_lab')
LAB_DIR.mkdir(exist_ok=True)
os.chdir(LAB_DIR)

print(f"📁 Lab directory: {LAB_DIR}")
print(f"📍 Current directory: {os.getcwd()}")

# Install dependencies
print("\n[*] Installing dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                'requests', 'paramiko', 'cryptography', 'flask', 'flask-cors'],
               check=False)

print("✅ Environment setup complete")

## STEP 2: Download/Create Lab Files

In [ ]:
# Option A: Clone from GitHub
print("[A] Clone from GitHub or [B] Create files inline?\n")
print("[A] Cloning repository...")

try:
    import urllib.request
    import json
    
    # List of files to download
    files_to_download = [
        'kvm_lab_server.py',
        'kvm_lab_cli.py',
        'ui.html',
        'requirements.txt',
        'kvm_lab.sh'
    ]
    
    base_url = "https://raw.githubusercontent.com/netanelcyber/HAMIVTZAR/claude/cve-2026-24858-fortios-sso-p1lrtu/cve-research/CVE-SUSPECTED-FORTIOS8-SSO/"
    
    print(f"📥 Downloading {len(files_to_download)} files...\n")
    
    for filename in files_to_download:
        try:
            url = base_url + filename
            print(f"   Downloading {filename}...", end=' ')
            urllib.request.urlretrieve(url, filename)
            print("✓")
        except Exception as e:
            print(f"✗ ({e})")
    
    print(f"\n✅ Files downloaded to {LAB_DIR}")
    print(f"\n📋 Files present:")
    for f in sorted(LAB_DIR.glob('*')):
        size = f.stat().st_size / 1024
        print(f"   - {f.name:25} ({size:.1f} KB)")
        
except Exception as e:
    print(f"⚠️  Download failed: {e}")
    print(f"\nCreating files inline instead...")

## STEP 3: Start HTTP Service

In [ ]:
import subprocess
import time

print("🚀 Starting KVM lab HTTP service...\n")

# Configuration
env = os.environ.copy()
env['KVM_HOST_IP'] = 'localhost'
env['KVM_HOST_USER'] = 'root'
env['FORTIGATE_SERIAL'] = 'FG-LAB-000001'
env['API_KEY'] = 'colab-service-key'
env['PORT'] = '5000'
env['DEBUG'] = 'False'

# Start service in background
print("[*] Launching service on 0.0.0.0:5000")
try:
    service_process = subprocess.Popen(
        [sys.executable, 'kvm_lab_server.py'],
        cwd=LAB_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    print("[*] Waiting for service to start...")
    time.sleep(3)
    
    # Check if running
    if service_process.poll() is None:
        print("✅ Service started (PID: {})\n".format(service_process.pid))
        print("🌐 Access points:")
        print("   Web UI: http://localhost:5000/")
        print("   API:    http://localhost:5000/api/*")
    else:
        stdout, stderr = service_process.communicate()
        print(f"❌ Service failed to start")
        print(f"Error: {stderr.decode()}")
        
except Exception as e:
    print(f"❌ Failed to start service: {e}")

# Store for later use
%store service_process

## STEP 4: Test Service

In [ ]:
import requests
import json

print("🔍 Testing service connection...\n")

try:
    # Test health endpoint
    response = requests.get('http://localhost:5000/health', timeout=5)
    result = response.json()
    
    print("✅ Service is running!\n")
    print("📊 Lab Status:")
    state = result.get('state', {})
    print(f"   KVM Connected:   {'✓' if state.get('kvm_connected') else '✗'}")
    print(f"   Tunnel Running:  {'✓' if state.get('tunnel_running') else '✗'}")
    print(f"   Exploit Success: {'✓' if state.get('exploit_results', {}).get('success') else '✗'}")
    
except Exception as e:
    print(f"❌ Service not responding: {e}")
    print(f"\nCheck service output above for errors")

## STEP 5: Terminal Access via CLI

In [ ]:
# Make CLI executable
os.chmod('kvm_lab_cli.py', 0o755)
os.chmod('kvm_lab.sh', 0o755)

print("🖥️  Terminal Access Ready\n")
print("Use these commands in the cells below:\n")
print("Python CLI:")
print("  python3 kvm_lab_cli.py --url http://localhost:5000 <command>\n")
print("Shell Script:")
print("  bash kvm_lab.sh <command>\n")
print("Available commands:")
print("  connect    - Connect to KVM host")
print("  setup      - Setup FortiOS 8")
print("  tunnel-start - Start SSH tunnel")
print("  exploit    - Run exploitation")
print("  recon      - Post-exploitation recon")
print("  cleanup    - Cleanup resources")
print("  full       - Run complete workflow")

## STEP 6: Lab Control - Python CLI

In [ ]:
# Check health
!python3 kvm_lab_cli.py --url http://localhost:5000 health

In [ ]:
# Get status
!python3 kvm_lab_cli.py --url http://localhost:5000 status

In [ ]:
# Connect to KVM
!python3 kvm_lab_cli.py --url http://localhost:5000 connect

In [ ]:
# Setup FortiOS 8
!python3 kvm_lab_cli.py --url http://localhost:5000 setup

In [ ]:
# Start tunnel
!python3 kvm_lab_cli.py --url http://localhost:5000 tunnel-start

In [ ]:
# Run exploitation
!python3 kvm_lab_cli.py --url http://localhost:5000 exploit

In [ ]:
# Post-exploitation recon
!python3 kvm_lab_cli.py --url http://localhost:5000 recon

In [ ]:
# Cleanup
!python3 kvm_lab_cli.py --url http://localhost:5000 cleanup

## STEP 7: Lab Control - Shell Script

In [ ]:
# Run full workflow with shell script
import os
os.environ['KVM_LAB_URL'] = 'http://localhost:5000'

!bash kvm_lab.sh help

In [ ]:
# Run full automated workflow
!bash kvm_lab.sh full

## STEP 8: Direct Bash Terminal

In [ ]:
# List files in lab directory
!ls -lh

In [ ]:
# View running processes
!ps aux | grep -E 'kvm_lab|python' | grep -v grep

In [ ]:
# Test service with curl
!curl -s http://localhost:5000/health | python3 -m json.tool

In [ ]:
# View service logs (if running in background)
import subprocess

try:
    result = subprocess.run(
        'curl -s http://localhost:5000/api/state | python3 -m json.tool',
        shell=True,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
except Exception as e:
    print(f"Error: {e}")

## STEP 9: Python Client Library

In [ ]:
# Use Python requests directly
import requests
import json

BASE_URL = "http://localhost:5000"
API_KEY = "colab-service-key"
headers = {'X-API-Key': API_KEY, 'Content-Type': 'application/json'}

print("Python Client Examples:\n")

# Example 1: Get config
print("[1] Get Configuration:")
response = requests.get(f"{BASE_URL}/api/config", headers=headers)
if response.status_code == 200:
    print(json.dumps(response.json(), indent=2))

# Example 2: Get state
print("\n[2] Get Lab State:")
response = requests.get(f"{BASE_URL}/api/state", headers=headers)
if response.status_code == 200:
    print(json.dumps(response.json(), indent=2))

## STEP 10: File Viewer

In [ ]:
# View available files
print("📋 Lab Files:\n")
for f in sorted(LAB_DIR.glob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  {f.name:30} {size:8.1f} KB")

In [ ]:
# View Python CLI code
with open('kvm_lab_cli.py', 'r') as f:
    lines = f.readlines()
    print(f"# kvm_lab_cli.py ({len(lines)} lines)")
    print(''.join(lines[:50]))  # First 50 lines
    print(f"\n... ({len(lines) - 50} more lines)")

In [ ]:
# View UI HTML
with open('ui.html', 'r') as f:
    content = f.read()
    print(f"# ui.html ({len(content)} bytes)")
    print(content[:500])  # First 500 characters
    print(f"\n... ({len(content) - 500} more characters)")

## STEP 11: Cleanup (When Done)

In [ ]:
# Stop service
print("🛑 Stopping service...\n")

try:
    %recall service_process
    if service_process.poll() is None:
        service_process.terminate()
        service_process.wait(timeout=5)
        print("✅ Service stopped")
    else:
        print("⚠️  Service already stopped")
except:
    print("⚠️  Could not stop service")

# Alternative: kill via bash
!pkill -f kvm_lab_server || true
print("\n✅ All cleanup complete")

## Notes

### Access Points
- **Web UI:** http://localhost:5000/ (open in browser)
- **API:** http://localhost:5000/api/* (with X-API-Key header)
- **CLI:** `python3 kvm_lab_cli.py` commands
- **Shell:** `bash kvm_lab.sh` commands
- **Terminal:** Direct bash commands with `!`

### Running Lab
- Service runs in background within Colab
- All files loaded into `/content/fortios8_kvm_lab/`
- Use any combination of Python, Shell, or Bash commands
- Service available for entire Colab session

### Requirements
- Lab service must be running (started in STEP 3)
- Actual KVM host not required for testing API
- For real exploitation: need actual KVM lab + SSH access

### Troubleshooting
- Check service in STEP 3 for errors
- Run `curl http://localhost:5000/health` to test
- View processes: `!ps aux | grep kvm`
- Kill service: `!pkill -f kvm_lab_server`

---

**Created:** July 23, 2026  
**Status:** Lab-Tested  
**Usage:** Google Colab Environment  
**Security:** Authorized Testing Only